# M5.4 Percolation and Random Resistor Networks

Book pointer: chapter 10, section 10.4 (current distributions in disordered media).

## 1. Recall

Answer on paper before running any cell.

1. Define bond percolation on a square lattice. What is the percolation threshold $p_c$ for bonds, and what changes in the lattice as $p$ crosses it?
2. Distinguish the spanning cluster, its backbone and its dead ends. Which of them carry current between two electrodes, and why do the others not?
3. State the effective-medium estimate of the conductivity of a bond-diluted square lattice. Where is it good and where does it fail?
4. Why does a single random lattice not give "the" conductivity at a given $p$? What do you average, and how do you report the uncertainty?
5. Write Kirchhoff's current law at a lattice node with unit-conductance bonds to its neighbours. What does it say about the node's potential?

## 2. Principle to algorithm

A disordered conductor (a porous rock with brine in its pores, a composite of metal grains in an insulator) is modelled as a square lattice of $n\times n$ nodes in which each bond between neighbours is present with probability $p$ and then has unit conductance, or is missing. Node $(i, j)$ sits at $x = i$, $y = j$. The left column $i = 0$ is an electrode held at $V = 1$, the right column $i = n-1$ an electrode at $V = 0$, and the top and bottom edges are open.

Two questions are asked of each lattice, and each has its own algorithm.

- **Does it span?** Treat the present bonds as the edges of a graph and label its connected components (`scipy.sparse.csgraph.connected_components`). The lattice spans if some label appears in both the left and the right column. `percolation.spans` does exactly this. Nothing about currents is needed.
- **How well does it conduct?** Kirchhoff's current law at every free node, $\sum_{\text{nb}} G_{\text{bond}}(V_{\text{nb}} - V_i) = 0$, is one linear equation per node, the same sparse system as the Laplace stencil of module 2 with missing bonds removed. `percolation.solve_lattice` hands it to `circuits.solve_network`, which also finds clusters that touch neither electrode and holds them at $V = 0$: they carry no current, and leaving them free would make the matrix singular. The result is a potential at every node and a current $I = G\,\Delta V$ in every bond.

The full lattice is $n$ parallel rows of $n - 1$ unit bonds in series, with conductance $n/(n-1)$. `percolation.effective_conductivity` divides the measured conductance by that number, so the full lattice gives exactly 1 and a lattice that does not span gives 0 (to round-off).

The current divides the spanning cluster into two parts. The **backbone** is the set of bonds that carry current: every one of them lies on some path from one electrode to the other. The **dead ends** are bonds attached to the cluster at a single point; whatever enters must leave by the same point, so their current is zero. `percolation.backbone` marks bonds whose current exceeds a tiny fraction of the largest.

**The effective-medium estimate.** Replace the random lattice by a uniform one of bond conductance $\sigma_m$ and demand that, on average, putting one real bond back into the uniform medium causes no extra potential drop. For a lattice where each node has $z$ neighbours this gives $\sigma_m = (p - 2/z)/(1 - 2/z)$, and for the square lattice ($z = 4$)

$$\sigma_{\text{EMA}} = 2\left(p - \tfrac12\right), \qquad p \ge \tfrac12.$$

It predicts the threshold $p_c = 1/2$ correctly (that value is exact for bond percolation on the square lattice, by a duality argument) and is accurate well above it. Close to $p_c$ the true conductivity vanishes more slowly than linearly, $\sigma \propto (p - p_c)^t$ with $t \approx 1.3$ in two dimensions, because the backbone near threshold is a sparse, tortuous web that no average medium describes.

Every number here is random. A single lattice gives one sample of $\sigma$; the notebook averages many lattices at each $p$ and reports the mean with its standard error, $s/\sqrt{N}$.

### Animation

A bond lattice between two electrodes while $p$ rises from 0.3 to 0.7. Bonds appear one by one; below about $p = 0.5$ there is no connected path and no current. When a spanning cluster forms, current starts to flow: bonds are coloured by the size of their current, the backbone lights up and the dead ends stay grey. Watch the conductivity readout leave zero only when the first path connects the two sides.

In [ ]:
from emlib import show
show.animation("m5_circuits", "PercolationBackbone")

## 3. Build

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
import scipy.sparse as sps
from scipy.sparse.csgraph import connected_components
from emlib import percolation as pc
from emlib.circuits import grid_edges

plt.rcParams["figure.figsize"] = (6, 5)
np.set_printoptions(precision=4, suppress=True)
T_START = time.time()


def bond_segments(n):
    '''Segment end points of every possible bond, in the order (h_bonds, v_bonds).'''
    i, j = np.meshgrid(np.arange(n - 1), np.arange(n), indexing="ij")
    h = np.stack([np.stack([i, j], -1), np.stack([i + 1, j], -1)], axis=-2).reshape(-1, 2, 2)
    i, j = np.meshgrid(np.arange(n), np.arange(n - 1), indexing="ij")
    v = np.stack([np.stack([i, j], -1), np.stack([i, j + 1], -1)], axis=-2).reshape(-1, 2, 2)
    return h, v


def spanning_cluster_bonds(h_bonds, v_bonds):
    '''Masks (same shapes as the bond arrays) of the present bonds that belong to a spanning cluster.'''
    n = h_bonds.shape[1]
    he, ve = grid_edges(n, n)
    edges = np.concatenate([he.reshape(-1, 2), ve.reshape(-1, 2)])
    present = np.concatenate([h_bonds.ravel(), v_bonds.ravel()])
    a, b = edges[present, 0], edges[present, 1]
    _, label = connected_components(sps.coo_matrix((np.ones(len(a)), (a, b)), shape=(n * n, n * n)), directed=False)
    span_labels = np.intersect1d(label.reshape(n, n)[0, :], label.reshape(n, n)[-1, :])
    in_span = np.isin(label[edges[:, 0]], span_labels) & present
    return in_span[: h_bonds.size].reshape(h_bonds.shape), in_span[h_bonds.size :].reshape(v_bonds.shape)


def kirchhoff_residual(Ih, Iv):
    '''Net current leaving each node; must vanish everywhere except on the two electrode columns.'''
    n = Ih.shape[1]
    net = np.zeros((n, n))
    net[:-1, :] += Ih
    net[1:, :] -= Ih
    net[:, :-1] += Iv
    net[:, 1:] -= Iv
    return net

### Three lattices around the threshold

The three lattices below use the same random numbers: each bond is drawn once as a uniform number $u$ and is present when $u < p$. Raising $p$ therefore only adds bonds, so the pictures show one lattice filling in. Bonds of a spanning cluster are drawn in colour, other present bonds in grey.

In [ ]:
n_pic, seed_pic = 32, 7
segH, segV = bond_segments(n_pic)
fig, axes = plt.subplots(1, 3, figsize=(15, 5.6))
for ax, p in zip(axes, (0.4, 0.5, 0.6)):
    h, v = pc.random_bonds(n_pic, p, rng=seed_pic)
    sh, sv = spanning_cluster_bonds(h, v)
    other = np.concatenate([segH[(h & ~sh).ravel()], segV[(v & ~sv).ravel()]])
    span = np.concatenate([segH[sh.ravel()], segV[sv.ravel()]])
    ax.add_collection(LineCollection(other, colors="0.75", linewidths=0.8))
    ax.add_collection(LineCollection(span, colors="C3", linewidths=1.4))
    ax.plot([0, 0], [0, n_pic - 1], "k", lw=4); ax.plot([n_pic - 1] * 2, [0, n_pic - 1], "k", lw=4)
    ax.set_xlim(-1, n_pic); ax.set_ylim(-1, n_pic); ax.set_aspect("equal"); ax.axis("off")
    ax.set_title(f"p = {p}: spans = {pc.spans(h, v)}\n{sh.sum() + sv.sum()} of {h.sum() + v.sum()} bonds in a spanning cluster", fontsize=11)
plt.show()

### Spanning probability

For each lattice size $n$ and each $p$, draw many lattices and count the fraction that span. On an infinite lattice the answer is a step: 0 below $p_c$ and 1 above. On a finite lattice the step is smeared over a window whose width shrinks as $n$ grows, and the curves for different $n$ cross near $p_c$. 400 lattices per point keep the statistical error of each fraction below 2.5 percent; the whole sweep takes a few seconds because spanning only needs connected components.

In [ ]:
ps_span = np.round(np.arange(0.30, 0.7001, 0.025), 3)
sizes, n_samples_span = (16, 32, 64), 400
P_span = {}
t0 = time.time()
for n in sizes:
    P_span[n] = np.array([np.mean([pc.spans(*pc.random_bonds(n, p, rng=100_000 * n + 1000 * k + s)) for s in range(n_samples_span)])
                          for k, p in enumerate(ps_span)])
print(f"{len(sizes) * len(ps_span) * n_samples_span} lattices in {time.time() - t0:.1f} s")
for n in sizes:
    print(f"n = {n:2d}: spanning fraction at p = 0.45, 0.50, 0.55: {P_span[n][ps_span == 0.45][0]:.2f}, {P_span[n][ps_span == 0.5][0]:.2f}, {P_span[n][ps_span == 0.55][0]:.2f}")

for n in sizes:
    plt.plot(ps_span, P_span[n], "o-", ms=4, label=f"n = {n}")
plt.axvline(pc.P_C_BOND_SQUARE, color="grey", ls=":", label="p_c = 1/2")
plt.xlabel("bond probability p"); plt.ylabel("fraction of lattices that span"); plt.legend(); plt.title("the step sharpens with lattice size"); plt.show()

The curves cross close to $p = 0.5$, where roughly half the lattices of every size span, and the transition window narrows with $n$: on the largest lattice the fraction goes from almost 0 to almost 1 between $p = 0.45$ and $0.55$.

### Effective conductivity

Now the Kirchhoff solve. At each $p$ from 0.40 to 0.90, 60 lattices of $64\times64$ nodes are solved, about 4 ms each. The figure shows the mean conductivity with its standard error, the effective-medium line, and the fraction of those same lattices that span. The individual samples are kept for the checks in section 4.

In [ ]:
n_sig, n_samples_sig = 64, 60
ps_sig = np.round(np.arange(0.40, 0.9001, 0.025), 3)
sig_samples = np.zeros((len(ps_sig), n_samples_sig))
span_samples = np.zeros((len(ps_sig), n_samples_sig), dtype=bool)
t0 = time.time()
for k, p in enumerate(ps_sig):
    for s in range(n_samples_sig):
        h, v = pc.random_bonds(n_sig, p, rng=7_000_000 + 1000 * k + s)
        span_samples[k, s] = pc.spans(h, v)
        sig_samples[k, s] = pc.effective_conductivity(h, v)
print(f"{sig_samples.size} Kirchhoff solves in {time.time() - t0:.1f} s")
sig_mean = sig_samples.mean(axis=1)
sig_err = sig_samples.std(axis=1, ddof=1) / np.sqrt(n_samples_sig)
ema = np.clip(2 * (ps_sig - 0.5), 0, None)

print("   p     mean sigma    std error    EMA     spanning fraction")
for p, m, e, a, f in zip(ps_sig, sig_mean, sig_err, ema, span_samples.mean(axis=1)):
    if np.isclose(p * 20, round(p * 20)):
        print(f"{p:5.2f}   {m:10.4f}   {e:10.4f}   {a:5.2f}   {f:8.2f}")

fig, ax = plt.subplots(figsize=(7, 5))
ax.errorbar(ps_sig, sig_mean, yerr=sig_err, fmt="o", ms=4, capsize=2, label=f"Kirchhoff, n = {n_sig}, {n_samples_sig} lattices per point")
ax.plot(ps_sig, ema, "k--", label="effective medium 2(p - 1/2)")
ax.plot(ps_sig, span_samples.mean(axis=1), "C2:", label="spanning fraction")
ax.axvline(0.5, color="grey", lw=0.8)
ax.set_xlabel("bond probability p"); ax.set_ylabel("sigma_eff / sigma_full"); ax.legend(); ax.set_title("conductivity of a bond-diluted lattice"); plt.show()

Three regimes are visible. Below $p \approx 0.45$ no lattice spans and the conductivity is zero. Above $p \approx 0.75$ the effective-medium line follows the data to within about 0.03: a dilute set of missing bonds is a small perturbation of the full lattice. In between, the conductivity rises from zero with upward curvature. At $p = 0.5$ it lies above the straight line, because a finite lattice spans by chance where the infinite one would not conduct at all. From $p = 0.55$ to $0.75$ it lies below the line: the backbone is a sparse, winding web that conducts worse than the averaged medium predicts. The true law near threshold is the power $(p - p_c)^{1.3}$, not a straight line.

### Current in one lattice: backbone and dead ends

One lattice at $p = 0.55$, a little above threshold, solved and drawn with each bond coloured by $\log_{10}$ of its current as a fraction of the total current. Present bonds that carry no current are grey. The seed is the first one that gives a spanning lattice.

In [ ]:
n_one, p_one = 64, 0.55
seed_one = next(s for s in range(1000) if pc.spans(*pc.random_bonds(n_one, p_one, rng=s)))
h1, v1 = pc.random_bonds(n_one, p_one, rng=seed_one)
V1, Ih1, Iv1, I1 = pc.solve_lattice(h1, v1)
bh1, bv1 = pc.backbone(Ih1, Iv1)
sh1, sv1 = spanning_cluster_bonds(h1, v1)

segH, segV = bond_segments(n_one)
I_all = np.concatenate([np.abs(Ih1).ravel(), np.abs(Iv1).ravel()]) / I1
on_bb = np.concatenate([bh1.ravel(), bv1.ravel()])
present = np.concatenate([h1.ravel(), v1.ravel()])
segs = np.concatenate([segH, segV])

fig, ax = plt.subplots(figsize=(8, 7.5))
ax.add_collection(LineCollection(segs[present & ~on_bb], colors="0.8", linewidths=0.7))
lc = LineCollection(segs[on_bb], cmap="inferno_r", linewidths=1.6)
lc.set_array(np.log10(I_all[on_bb])); lc.set_clim(-4, 0)
ax.add_collection(lc)
plt.colorbar(lc, ax=ax, label="log10 (bond current / total current)")
ax.plot([0, 0], [0, n_one - 1], "C3", lw=4); ax.plot([n_one - 1] * 2, [0, n_one - 1], "C0", lw=4)
ax.set_xlim(-1, n_one); ax.set_ylim(-1, n_one); ax.set_aspect("equal"); ax.axis("off")
ax.set_title(f"p = {p_one}, seed {seed_one}: backbone coloured by current, dead ends and isolated clusters grey")
plt.show()

n_present, n_span, n_bb = present.sum(), sh1.sum() + sv1.sum(), on_bb.sum()
print(f"present bonds {n_present}, in the spanning cluster {n_span} ({n_span / n_present:.2f}), on the backbone {n_bb} ({n_bb / n_present:.2f})")
print(f"sigma_eff = {I1 * (n_one - 1) / n_one:.4f}; largest single bond current = {I_all.max():.3f} of the total")

Only part of the present bonds belongs to the spanning cluster, and only part of that cluster is backbone. The rest are clusters that touch at most one electrode and dead ends hanging off the backbone; they are part of the material but play no part in conduction. Along the backbone the current is far from uniform: a few single bonds that every path must cross (the darkest segments) carry a large share of the total, while long detours carry thousandths of it.

### Distribution of bond currents

The histogram counts backbone bonds by the logarithm of their current, for the lattice above and for one at $p = 0.8$. Far above threshold the current spreads over many parallel paths and the distribution is narrow. Near threshold it spans several decades: most backbone bonds carry little, and a small number of bottleneck bonds carry the bulk of the current.

In [ ]:
h8, v8 = pc.random_bonds(n_one, 0.8, rng=seed_one)
_, Ih8, Iv8, I8 = pc.solve_lattice(h8, v8)
bh8, bv8 = pc.backbone(Ih8, Iv8)
I_bb8 = np.concatenate([np.abs(Ih8)[bh8], np.abs(Iv8)[bv8]]) / I8
I_bb1 = I_all[on_bb]

bins = np.linspace(-8, 0, 49)
plt.hist(np.log10(I_bb1), bins=bins, alpha=0.6, label=f"p = {p_one} ({len(I_bb1)} backbone bonds)")
plt.hist(np.log10(I_bb8), bins=bins, alpha=0.6, label=f"p = 0.8 ({len(I_bb8)} backbone bonds)")
plt.xlabel("log10 (bond current / total current)"); plt.ylabel("number of bonds"); plt.legend(); plt.title("bond currents on the backbone"); plt.show()
for name, I_b in ((f"p = {p_one}", I_bb1), ("p = 0.8", I_bb8)):
    srt = np.sort(I_b ** 2)[::-1]
    k_half = int(np.searchsorted(np.cumsum(srt), 0.5 * srt.sum())) + 1
    print(f"{name}: decades spanned (1st to 99th percentile) {np.log10(np.percentile(I_b, 99) / np.percentile(I_b, 1)):.1f}; "
          f"half the dissipated power is in {k_half} of {len(I_b)} backbone bonds ({k_half / len(I_b):.3f})")

The power dissipated in a unit bond is $I^2$, so the last line counts how few bonds carry half of the Joule heat. Near threshold that fraction is small: the heat, and with it the risk of a bond burning out, is concentrated in the bottlenecks. Far above threshold it is spread over a large share of the backbone.

### Backbone fraction versus $p$

For spanning lattices at each $p$, the fraction of present bonds that belong to the spanning cluster and the fraction that lie on its backbone. 20 lattices of $64\times64$ per point.

In [ ]:
ps_bb = np.round(np.arange(0.50, 0.9001, 0.05), 3)
frac_span, frac_bb = [], []
for k, p in enumerate(ps_bb):
    fs, fb = [], []
    for s in range(20):
        h, v = pc.random_bonds(n_one, p, rng=9_000_000 + 1000 * k + s)
        if not pc.spans(h, v):
            continue
        _, Ih, Iv, _ = pc.solve_lattice(h, v)
        bh, bv = pc.backbone(Ih, Iv)
        sh, sv = spanning_cluster_bonds(h, v)
        n_p = h.sum() + v.sum()
        fs.append((sh.sum() + sv.sum()) / n_p); fb.append((bh.sum() + bv.sum()) / n_p)
    frac_span.append(np.mean(fs)); frac_bb.append(np.mean(fb))
    print(f"p = {p:.2f}: {len(fs):2d} spanning lattices, spanning-cluster fraction {frac_span[-1]:.3f}, backbone fraction {frac_bb[-1]:.3f}")

plt.plot(ps_bb, frac_span, "o-", label="in the spanning cluster")
plt.plot(ps_bb, frac_bb, "s-", label="on the backbone")
plt.xlabel("bond probability p"); plt.ylabel("fraction of present bonds"); plt.ylim(0, 1.02); plt.legend(); plt.title("dead ends and isolated clusters shrink as p grows"); plt.show()

Near threshold the spanning cluster holds only part of the present bonds and the backbone a smaller part still. At $p = 0.5$ more than half of the spanning cluster is dead ends; by $p = 0.6$ three quarters of it carries current. As $p$ grows, isolated clusters merge into the spanning one and loops close up dead ends, until at high $p$ almost every bond carries current. The conductivity curve above is the combined result of these two effects and of the tortuosity of the backbone paths.

### Exercise: spanning fraction

Replace the body of `spanning_fraction` with your own version: draw `samples` lattices of $n\times n$ nodes at bond probability `p` with `pc.random_bonds`, using the seeds `seed, seed + 1, ...`, and return the fraction that span. The check runs it well below and well above threshold, where the answers must be close to 0 and 1.

In [ ]:
def spanning_fraction(n, p, samples, seed=0):
    '''Fraction of `samples` random n x n bond lattices at probability p that connect left to right.'''
    hits = 0                                                     # replace from here with your own version
    for s in range(samples):
        hits += pc.spans(*pc.random_bonds(n, p, rng=seed + s))
    return hits / samples


f_low, f_high = spanning_fraction(32, 0.2, 200, seed=11), spanning_fraction(32, 0.8, 200, seed=11)
print(f"n = 32, 200 lattices: spanning fraction {f_low:.3f} at p = 0.2 and {f_high:.3f} at p = 0.8")
assert f_low <= 0.02 and f_high >= 0.98

## 4. Verify

Five checks with closed forms or exact conservation laws.

1. The full lattice has conductivity exactly 1 by construction of the normalisation.
2. Kirchhoff's current law: in the $p = 0.55$ lattice the net current out of every node off the electrodes is zero, relative to the total current.
3. Energy: with unit bonds and 1 V across the lattice the power drawn from the electrodes, $I_{\text{total}}\cdot 1\,\text{V}$, equals the Joule heat $\sum I_{\text{bond}}^2$ dissipated in the bonds.
4. Effective medium: at $p = 0.8$ and $0.9$ the mean conductivity agrees with $2(p - 1/2)$ to within 0.03. The residual is the effective-medium approximation's own error plus a standard error near 0.002.
5. Topology: every sample of the conductivity sweep that did not span had zero conductivity, and every sample that did span had conductivity above zero. "Zero" here means round-off: a cluster that touches only the left electrode sits at $V = 1$ to within the solver's precision, so the current printed for it is of order $10^{-14}$, and the check allows $10^{-12}$.

In [ ]:
full = np.ones((n_sig - 1, n_sig), bool), np.ones((n_sig, n_sig - 1), bool)
sig_full = pc.effective_conductivity(*full)
res = kirchhoff_residual(Ih1, Iv1)[1:-1, :]
joule = (Ih1 ** 2).sum() + (Iv1 ** 2).sum()
ema_dev = {p: abs(sig_mean[np.isclose(ps_sig, p)][0] - 2 * (p - 0.5)) for p in (0.8, 0.9)}
nonspan_max = np.abs(sig_samples[~span_samples]).max()
span_min = sig_samples[span_samples].min()

print(f"1. full lattice: sigma_eff = {sig_full:.12f}")
print(f"2. Kirchhoff: max |net current| off the electrodes / total current = {np.abs(res).max() / I1:.1e}")
print(f"3. energy: I_total x 1 V = {I1:.6f}, sum of I^2 = {joule:.6f}, relative difference {abs(joule - I1) / I1:.1e}")
print(f"4. EMA: |mean sigma - 2(p - 1/2)| = {ema_dev[0.8]:.4f} at p = 0.8, {ema_dev[0.9]:.4f} at p = 0.9")
print(f"5. {(~span_samples).sum()} non-spanning samples, largest |sigma| among them {nonspan_max:.1e}; "
      f"{span_samples.sum()} spanning samples, smallest sigma among them {span_min:.4f}")

assert np.isclose(sig_full, 1.0, atol=1e-12)
assert np.abs(res).max() < 1e-10 * I1
assert abs(joule - I1) < 1e-10 * I1
assert ema_dev[0.8] < 0.03 and ema_dev[0.9] < 0.03
assert nonspan_max < 1e-12 and span_min > 1e-6
print(f"\ntotal notebook compute time so far: {time.time() - T_START:.1f} s")

## 5. Explore and rebuild

Explore. Near threshold the conductivity of an infinite lattice follows $\sigma \propto (p - p_c)^t$ with $t \approx 1.3$ in two dimensions. Estimate $t$ yourself: at $n = 64$, average the conductivity over at least 100 lattices at each of six or more values of $p$ between 0.55 and 0.70, and fit a straight line to $\log\sigma$ against $\log(p - 1/2)$. Compare the slope with 1.3. Then think about which way the finite lattice biases the fit: close to $p_c$ a $64\times64$ lattice spans more easily than an infinite one, so $\sigma$ is too large there and the slope comes out too small; far from $p_c$ the power law itself stops holding. Repeat at $n = 32$ and $n = 128$ and see whether the slope moves towards 1.3 as $n$ grows, and how the best fitting window shifts. What would you need to do to get $t$ to two digits?

Rebuild. Close this notebook. In a blank editor, write the lattice assembly from scratch: number the nodes, list the horizontal and vertical bonds, build the sparse Kirchhoff matrix for the free nodes with the two electrode columns moved to the right-hand side, hold clusters that touch no electrode at zero, solve, and return the bond currents. Then write the spanning test with connected components. Check both against `percolation.solve_lattice` and `percolation.spans` on a few random lattices.